In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# === Step 1: Load and merge the two CSVs ===
df1 = pd.read_csv('../Dashboard_Final/avg_gap_scores.csv')   # contains gap_score
df2 = pd.read_csv('../zipcodes_income.csv')                 # contains Zipcodes, Income

# Clean and align columns
df2.rename(columns={'Zipcodes': 'zipcode'}, inplace=True)
df2['zipcode'] = df2['zipcode'].astype(str)
df1['zipcode'] = df1['zipcode'].astype(str)

# Drop rows with missing data
df1 = df1.dropna(subset=['gap_score'])
df2 = df2.dropna(subset=['Income'])

# Merge on zipcode
merged_df = pd.merge(df1[['zipcode', 'gap_score']], df2, on='zipcode', how='inner')

# === Step 2: Feature selection and scaling ===
features = merged_df[['gap_score', 'Income']]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# === Step 3: Elbow method ===
inertia = []
sil_scores = []
K_range = range(2, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(features_scaled)
    inertia.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(features_scaled, labels))

# Plot Elbow
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(K_range, inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('k')
plt.ylabel('Inertia')

# Plot Silhouette
plt.subplot(1, 2, 2)
plt.plot(K_range, sil_scores, marker='o', color='orange')
plt.title('Silhouette Scores')
plt.xlabel('k')
plt.ylabel('Score')
plt.tight_layout()
plt.show()


In [ ]:
# Final KMeans clustering
k_optimal = 3
kmeans = KMeans(n_clusters=k_optimal, random_state=42)
merged_df['cluster'] = kmeans.fit_predict(features_scaled)


In [ ]:
for c in sorted(merged_df['cluster'].unique()):
    print(f"\nCluster {c} Zipcodes:")
    print(merged_df[merged_df['cluster'] == c]['zipcode'].tolist())


In [ ]:
summary = merged_df.groupby('cluster')[['gap_score', 'Income']].mean()
summary['count'] = merged_df.groupby('cluster')['zipcode'].count()
print(summary)



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 2D scatter plot of the clusters
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=merged_df,
    x='gap_score',
    y='Income',
    hue='cluster',
    palette='tab10',
    s=100,
    edgecolor='black'
)

# Plot cluster centers
centers = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(
    centers[:, 0], centers[:, 1],
    c='black', s=200, marker='X', label='Centroids'
)

plt.title(f'KMeans Clusters (k={k_optimal}) on Gap Score vs Income')
plt.xlabel('Gap Score')
plt.ylabel('Income')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
summary = merged_df.groupby('cluster')[['gap_score', 'Income']].median()
summary['count'] = merged_df.groupby('cluster')['zipcode'].count()
print(summary)

summary = merged_df.groupby('cluster')[['gap_score', 'Income']].mean()
summary['count'] = merged_df.groupby('cluster')['zipcode'].count()
print(summary)



